# Imports

In [11]:
import warnings
import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.feature_selection import VarianceThreshold, SelectKBest
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.exceptions import ConvergenceWarning
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from sklearn.model_selection import cross_validate, train_test_split, StratifiedKFold
from sklearn.metrics import f1_score, classification_report, confusion_matrix

In [12]:
# "default" to show warnings, "ignore" to hide warnings
warnings.filterwarnings("default", category=ConvergenceWarning)

In [13]:
SEED = 123

# Dataframe

In [5]:
df = pd.read_excel('./train.xlsx')
df.describe()

,resp_text,clarity
count,20092,20092
unique,18432,3
top,"Prezado (a) Senhor (a), Em resposta ao seu ...",c5
freq,81,6892


In [6]:
X = df["resp_text"].astype(str)
Y = df["clarity"]
print("X Y: ", X.shape, Y.shape)

X Y:  (20092,) (20092,)


In [7]:
x_train, x_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=SEED, stratify=Y)
print("x_train x_test: ", x_train.shape, x_test.shape)
print("y_train y_test: ", y_train.shape, y_test.shape)

x_train x_test:  (16073,) (4019,)
y_train y_test:  (16073,) (4019,)


# Dummy Majority

In [ ]:
vect = CountVectorizer()
x_train_vec = vect.fit_transform(x_train)
x_test_vec = vect.transform(x_test)

print("x_train_vec x_test_vec: ", x_train_vec.shape, x_test_vec.shape)
print()

clf = DummyClassifier(strategy='most_frequent', random_state = SEED, constant = None).fit(x_train_vec, y_train)

predicted = clf.predict(x_test_vec)
score = f1_score(y_test, predicted, average='macro')

print("------------ Dummy Majority ------------")
print("F1 SCORE MACRO AVERAGE:")
print(score)
print()

print("CONFUSION MATRIX:")
print(confusion_matrix(y_test, predicted, labels=["c1", "c234", "c5"]))
print()

print("CLASSIFICATION REPORT:")
print(classification_report(y_test, predicted, zero_division=0))

x_train_vec x_test_vec:  (16073, 46439) (4019, 46439)

------------ Dummy Majority ------------
F1 SCORE MACRO AVERAGE:
0.17030999135482275

CONFUSION MATRIX:
[[   0    0 1269]
 [   0    0 1371]
 [   0    0 1379]]

CLASSIFICATION REPORT:
              precision    recall  f1-score   support

          c1       0.00      0.00      0.00      1269
        c234       0.00      0.00      0.00      1371
          c5       0.34      1.00      0.51      1379

    accuracy                           0.34      4019
   macro avg       0.11      0.33      0.17      4019
weighted avg       0.12      0.34      0.18      4019



# Bow RegLog

In [ ]:
vect = CountVectorizer()
x_train_vec = vect.fit_transform(x_train)
x_test_vec = vect.transform(x_test)

print("x_train_vec x_test_vec: ", x_train_vec.shape, x_test_vec.shape)
print()

clf = LogisticRegression(class_weight='balanced', max_iter=100, random_state=SEED).fit(x_train_vec, y_train)

predicted = clf.predict(x_test_vec)
score = f1_score(y_test, predicted, average='macro')

print("------------ Bow RegLog ------------")
print("F1 SCORE MACRO AVERAGE:")
print(score)
print()

print("CONFUSION MATRIX:")
print(confusion_matrix(y_test, predicted, labels=["c1", "c234", "c5"]))
print()

print("CLASSIFICATION REPORT:")
print(classification_report(y_test, predicted))

x_train_vec x_test_vec:  (16073, 46439) (4019, 46439)

------------ Bow RegLog ------------
F1 SCORE MACRO AVERAGE:
0.4401628194583293

CONFUSION MATRIX:
[[639 306 324]
 [472 433 466]
 [328 338 713]]

CLASSIFICATION REPORT:
              precision    recall  f1-score   support

          c1       0.44      0.50      0.47      1269
        c234       0.40      0.32      0.35      1371
          c5       0.47      0.52      0.49      1379

    accuracy                           0.44      4019
   macro avg       0.44      0.45      0.44      4019
weighted avg       0.44      0.44      0.44      4019



# BoW TF-IDF RegLog

In [ ]:
vect = TfidfVectorizer()
x_train_vec = vect.fit_transform(x_train)
x_test_vec = vect.transform(x_test)

print("x_train_vec x_test_vec: ", x_train_vec.shape, x_test_vec.shape)
print()

clf = LogisticRegression(class_weight='balanced', max_iter=100, random_state=SEED).fit(x_train_vec, y_train)

predicted = clf.predict(x_test_vec)
score = f1_score(y_test, predicted, average='macro')

print("------------ BoW TF-IDF ------------")
print("F1 SCORE MACRO AVERAGE:")
print(score)
print()

print("CONFUSION MATRIX:")
print(confusion_matrix(y_test, predicted, labels=["c1", "c234", "c5"]))
print()

print("CLASSIFICATION REPORT:")
print(classification_report(y_test, predicted))

x_train_vec x_test_vec:  (16073, 46439) (4019, 46439)

------------ BoW TF-IDF ------------
F1 SCORE MACRO AVERAGE:
0.4647566493904364

CONFUSION MATRIX:
[[686 330 253]
 [450 459 462]
 [267 378 734]]

CLASSIFICATION REPORT:
              precision    recall  f1-score   support

          c1       0.49      0.54      0.51      1269
        c234       0.39      0.33      0.36      1371
          c5       0.51      0.53      0.52      1379

    accuracy                           0.47      4019
   macro avg       0.46      0.47      0.46      4019
weighted avg       0.46      0.47      0.46      4019



# BoW TF-IDF RegLog kbest fixed

In [36]:
vect = TfidfVectorizer()
x_train_vec = vect.fit_transform(x_train)
x_test_vec = vect.transform(x_test)

print("x_train_vec x_test_vec: ", x_train_vec.shape, x_test_vec.shape)

variance_filter = VarianceThreshold(threshold=0.0)
x_train_filt = variance_filter.fit_transform(x_train_vec)
x_test_filt = variance_filter.transform(x_test_vec)

print("x_train_filt x_test_filt: ", x_train_filt.shape, x_test_filt.shape)

kvalue = min(5000, x_train_filt.shape[1])
best_sel = SelectKBest(k=kvalue)
best_fit = best_sel.fit(x_train_filt, y_train)
x_train_best = best_fit.transform(x_train_filt)
x_test_best = best_fit.transform(x_test_filt)

print("x_train_best x_test_best:", x_train_best.shape, x_test_best.shape)
print()

clf = LogisticRegression(class_weight="balanced", max_iter=100, random_state=SEED).fit(x_train_best, y_train)

predicted = clf.predict(x_test_best)
score = f1_score(y_test, predicted, average="macro")

print("------------ BoW TF-IDF RegLog kbest fixed ------------")
print("kvalue: ", kvalue)
print()

print("F1 SCORE MACRO AVERAGE:")
print(score)
print()

print("CONFUSION MATRIX:")
print(confusion_matrix(y_test, predicted, labels=["c1", "c234", "c5"]))
print()

print("CLASSIFICATION REPORT:")
print(classification_report(y_test, predicted))

x_train_vec x_test_vec:  (16073, 46439) (4019, 46439)
x_train_filt x_test_filt:  (16073, 46439) (4019, 46439)
x_train_best x_test_best: (16073, 5000) (4019, 5000)

------------ BoW TF-IDF RegLog kbest fixed ------------
kvalue:  5000

F1 SCORE MACRO AVERAGE:
0.4572506465704999

CONFUSION MATRIX:
[[655 317 297]
 [470 451 450]
 [270 363 746]]

CLASSIFICATION REPORT:
              precision    recall  f1-score   support

          c1       0.47      0.52      0.49      1269
        c234       0.40      0.33      0.36      1371
          c5       0.50      0.54      0.52      1379

    accuracy                           0.46      4019
   macro avg       0.46      0.46      0.46      4019
weighted avg       0.46      0.46      0.46      4019



# BoW TF-IDF RegLog kbest searched

In [37]:
vect = TfidfVectorizer()
x_train_vec = vect.fit_transform(x_train)
x_test_vec = vect.transform(x_test)

print("x_train_vec x_test_vec: ", x_train_vec.shape, x_test_vec.shape)

# Remove constant features using training data only
variance_filter = VarianceThreshold(threshold=0.0)
x_train_filt = variance_filter.fit_transform(x_train_vec)
x_test_filt = variance_filter.transform(x_test_vec)

print("x_train_filt x_test_filt: ", x_train_filt.shape, x_test_filt.shape)
print()

print("------------ BoW TF-IDF RegLog kbest searched ------------")
bestScore = 0.0
bestK = 0
kvalues = list(range(x_train_filt.shape[1], 10000, -10000))
for kvalue in kvalues:
    best_sel = SelectKBest(k=kvalue)
    best_fit = best_sel.fit(x_train_filt, y_train)
    x_train_best = best_fit.transform(x_train_filt)
    x_test_best = best_fit.transform(x_test_filt)

    clf = LogisticRegression(class_weight="balanced", max_iter=100, random_state=SEED).fit(x_train_best, y_train)

    predicted = clf.predict(x_test_best)
    score = f1_score(y_test, predicted, average="macro")
    print("kvalue: ", kvalue)
    print("F1 SCORE MACRO AVERAGE:", score)
    print()
    
    if score > bestScore:
        bestScore = score
        bestK = kvalue

print("bestScore = ", bestScore)
print("bestK = ", bestK)

x_train_vec x_test_vec:  (16073, 46439) (4019, 46439)
x_train_filt x_test_filt:  (16073, 46439) (4019, 46439)

------------ BoW TF-IDF RegLog kbest searched ------------
kvalue:  46439
F1 SCORE MACRO AVERAGE: 0.4647566493904364

kvalue:  36439
F1 SCORE MACRO AVERAGE: 0.4649178352868352

kvalue:  26439
F1 SCORE MACRO AVERAGE: 0.4636863178024859

kvalue:  16439
F1 SCORE MACRO AVERAGE: 0.4651639864717941

bestScore =  0.4651639864717941
bestK =  16439


# BoW TF-IDF RegLog with range

In [ ]:
vect = TfidfVectorizer(ngram_range=(1, 3), analyzer='word', min_df=3, sublinear_tf=True)
x_train_vec = vect.fit_transform(x_train)
x_test_vec = vect.transform(x_test)

print("x_train_vec x_test_vec: ", x_train_vec.shape, x_test_vec.shape)
print()

clf = LogisticRegression(class_weight='balanced', max_iter=1000, solver='saga', random_state=SEED).fit(x_train_vec, y_train)

predicted = clf.predict(x_test_vec)
score = f1_score(y_test, predicted, average='macro')

print("------------ BoW TF-IDF RegLog with range ------------")
print("F1 SCORE MACRO AVERAGE:")
print(score)
print()

print("CONFUSION MATRIX:")
print(confusion_matrix(y_test, predicted, labels=["c1", "c234", "c5"]))
print()

print("CLASSIFICATION REPORT:")
print(classification_report(y_test, predicted))

x_train_vec x_test_vec:  (16073, 181794) (4019, 181794)

------------ BoW TF-IDF RegLog with range ------------
F1 SCORE MACRO AVERAGE:
0.4765150861439367

CONFUSION MATRIX:
[[666 336 267]
 [432 512 427]
 [240 397 742]]

CLASSIFICATION REPORT:
              precision    recall  f1-score   support

          c1       0.50      0.52      0.51      1269
        c234       0.41      0.37      0.39      1371
          c5       0.52      0.54      0.53      1379

    accuracy                           0.48      4019
   macro avg       0.48      0.48      0.48      4019
weighted avg       0.47      0.48      0.48      4019



# BoW TF-IDF RegLog K-Fold

In [ ]:
pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        ngram_range=(1, 3),
        analyzer="word",
        min_df=3,
        sublinear_tf=True,
    )),
    ("classifier", LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        solver="saga",
        random_state=SEED,
    )),
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
scores = cross_validate(pipeline, x_train, y_train, cv=cv, scoring="f1_macro", n_jobs=-1)

print("Fold scores:", scores["test_score"])
print("Mean macro-F1:", scores["test_score"].mean())
print("Std:", scores["test_score"].std())
print()

pipeline.fit(x_train, y_train)
predicted = pipeline.predict(x_test)
score = f1_score(y_test, predicted, average='macro')

print("------------ NGram TF-IDF RegLog Cross-Validation ------------")
print("F1 SCORE MACRO AVERAGE:")
print(score)
print()

print("CONFUSION MATRIX:")
print(confusion_matrix(y_test, predicted, labels=["c1", "c234", "c5"]))
print()

print("CLASSIFICATION REPORT:")
print(classification_report(y_test, predicted, zero_division=0))

------------ NGram TF-IDF RegLog Cross-Validation ------------
Fold scores: [0.45117949 0.44560907 0.44738049 0.45223754 0.43350168]
Mean macro-F1: 0.44598165326546935
Std: 0.006693263260336131

CONFUSION MATRIX:
[[673 321 275]
 [446 488 437]
 [255 385 739]]

CLASSIFICATION REPORT:
              precision    recall  f1-score   support

          c1       0.49      0.53      0.51      1269
        c234       0.41      0.36      0.38      1371
          c5       0.51      0.54      0.52      1379

    accuracy                           0.47      4019
   macro avg       0.47      0.47      0.47      4019
weighted avg       0.47      0.47      0.47      4019



# NGram TF-IDF RegLog

In [ ]:
vect = TfidfVectorizer(ngram_range=(3, 6), analyzer='char', min_df=3, sublinear_tf=True)
x_train_vec = vect.fit_transform(x_train)
x_test_vec = vect.transform(x_test)

print("x_train_vec x_test_vec: ", x_train_vec.shape, x_test_vec.shape)
print()

clf = LogisticRegression(class_weight='balanced', max_iter=1000, solver='saga', random_state=SEED).fit(x_train_vec, y_train)

predicted = clf.predict(x_test_vec)
score = f1_score(y_test, predicted, average='macro')

print("------------ NGram TF-IDF RegLog ------------")
print("F1 SCORE MACRO AVERAGE:")
print(score)
print()

print("CONFUSION MATRIX:")
print(confusion_matrix(y_test, predicted, labels=["c1", "c234", "c5"]))
print()

print("CLASSIFICATION REPORT:")
print(classification_report(y_test, predicted))

x_train_vec x_test_vec:  (16073, 459101) (4019, 459101)

------------ NGram TF-IDF RegLog ------------
F1 SCORE MACRO AVERAGE:
0.4706793586375635

CONFUSION MATRIX:
[[673 321 275]
 [446 488 437]
 [255 385 739]]

CLASSIFICATION REPORT:
              precision    recall  f1-score   support

          c1       0.49      0.53      0.51      1269
        c234       0.41      0.36      0.38      1371
          c5       0.51      0.54      0.52      1379

    accuracy                           0.47      4019
   macro avg       0.47      0.47      0.47      4019
weighted avg       0.47      0.47      0.47      4019



# NGram TF-IDF LinearSVC

In [ ]:
vect = TfidfVectorizer(ngram_range=(3, 6), analyzer='char', min_df=3, sublinear_tf=True)
x_train_vec = vect.fit_transform(x_train)
x_test_vec = vect.transform(x_test)

print("x_train_vec x_test_vec: ", x_train_vec.shape, x_test_vec.shape)
print()

clf = LinearSVC(class_weight="balanced", C=1.0, random_state=SEED).fit(x_train_vec, y_train)

predicted = clf.predict(x_test_vec)
score = f1_score(y_test, predicted, average='macro')

print("------------ NGram TF-IDF LinearSVC ------------")
print("F1 SCORE MACRO AVERAGE:")
print(score)
print()

print("CONFUSION MATRIX:")
print(confusion_matrix(y_test, predicted, labels=["c1", "c234", "c5"]))
print()

print("CLASSIFICATION REPORT:")
print(classification_report(y_test, predicted))

x_train_vec x_test_vec:  (16073, 459101) (4019, 459101)

------------ NGram TF-IDF LinearSVC ------------
F1 SCORE MACRO AVERAGE:
0.4587721672435959

CONFUSION MATRIX:
[[654 341 274]
 [448 495 428]
 [290 389 700]]

CLASSIFICATION REPORT:
              precision    recall  f1-score   support

          c1       0.47      0.52      0.49      1269
        c234       0.40      0.36      0.38      1371
          c5       0.50      0.51      0.50      1379

    accuracy                           0.46      4019
   macro avg       0.46      0.46      0.46      4019
weighted avg       0.46      0.46      0.46      4019



# NGram TF-IDF SGDC

In [ ]:
vect = TfidfVectorizer(ngram_range=(3, 6), analyzer='char', min_df=3, sublinear_tf=True)
x_train_vec = vect.fit_transform(x_train)
x_test_vec = vect.transform(x_test)

print("x_train_vec x_test_vec: ", x_train_vec.shape, x_test_vec.shape)
print()

clf = SGDClassifier(loss="log_loss", class_weight="balanced", max_iter=1000, random_state=SEED).fit(x_train_vec, y_train)

predicted = clf.predict(x_test_vec)
score = f1_score(y_test, predicted, average='macro')

print("------------ NGram TF-IDF SGDC ------------")
print("F1 SCORE MACRO AVERAGE:")
print(score)
print()

print("CONFUSION MATRIX:")
print(confusion_matrix(y_test, predicted, labels=["c1", "c234", "c5"]))
print()

print("CLASSIFICATION REPORT:")
print(classification_report(y_test, predicted))

x_train_vec x_test_vec:  (16073, 459101) (4019, 459101)

------------ NGram TF-IDF SGDC ------------
F1 SCORE MACRO AVERAGE:
0.4640205482353615

CONFUSION MATRIX:
[[597 323 349]
 [394 469 508]
 [213 348 818]]

CLASSIFICATION REPORT:
              precision    recall  f1-score   support

          c1       0.50      0.47      0.48      1269
        c234       0.41      0.34      0.37      1371
          c5       0.49      0.59      0.54      1379

    accuracy                           0.47      4019
   macro avg       0.47      0.47      0.46      4019
weighted avg       0.46      0.47      0.46      4019



# NGram TF-IDF LightGBM

In [14]:
vect = TfidfVectorizer(ngram_range=(3, 6), analyzer='char', min_df=3, sublinear_tf=True)
x_train_vec = vect.fit_transform(x_train)
x_test_vec = vect.transform(x_test)

print("x_train_vec x_test_vec: ", x_train_vec.shape, x_test_vec.shape)
print()

model = LGBMClassifier(n_estimators=300, learning_rate=0.05, class_weight='balanced', random_state=SEED).fit(x_train_vec, y_train)

predicted = model.predict(x_test_vec)
score = f1_score(y_test, predicted, average='macro')

print("------------ TF-IDF LightGBM ------------")
print("F1 SCORE MACRO AVERAGE:")
print(score)
print()

print("CONFUSION MATRIX:")
print(confusion_matrix(y_test, predicted, labels=["c1", "c234", "c5"]))
print()

print("CLASSIFICATION REPORT:")
print(classification_report(y_test, predicted))

x_train_vec x_test_vec:  (16073, 459101) (4019, 459101)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 17,514538 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 7658170
[LightGBM] [Info] Number of data points in the train set: 16073, number of used features: 139727
[LightGBM] [Info] Start training from score -1,098612
[LightGBM] [Info] Start training from score -1,098612
[LightGBM] [Info] Start training from score -1,098612
------------ TF-IDF LightGBM ------------
F1 SCORE MACRO AVERAGE:
0.44319523159425894

CONFUSION MATRIX:
[[627 353 289]
 [439 475 457]
 [289 406 684]]

CLASSIFICATION REPORT:
              precision    recall  f1-score   support

          c1       0.46      0.49      0.48      1269
        c234       0.38      0.35      0.36      1371
          c5       0.48      0.50      0.49      1379

    accuracy                           0.44      4019
   macro avg       0.44      0.45    

# BERTimbau

In [ ]:
import os
import torch
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

# ----------------------------------------------------------------------
# 1. CONFIGURAÇÕES E DISPOSITIVO
# ----------------------------------------------------------------------
MODEL_NAME = "neuralmind/bert-base-portuguese-cased"
MAX_LENGTH = 512
BATCH_SIZE = 16
EPOCHS = 3
LEARNING_RATE = 2e-5
SEED = 42

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Executando no dispositivo: {device}")

# ----------------------------------------------------------------------
# 2. PRÉ-PROCESSAMENTO DOS DADOS
# ----------------------------------------------------------------------
# Mapeamento de rótulos de texto para inteiros
label_map = {"c1": 0, "c234": 1, "c5": 2}
id2label = {0: "c1", 1: "c234", 2: "c5"}

# Remover eventuais linhas sem texto ou sem rótulo
df = df.dropna(subset=["resp_text", "clarity"]).copy()
df["label"] = df["clarity"].map(label_map)

# Divisão Treino (80%), Validação (10%) e Teste (10%) com estratificação
train_df, test_df = train_test_split(
    df, test_size=0.2, random_state=SEED, stratify=df["label"]
)
val_df, test_df = train_test_split(
    test_df, test_size=0.5, random_state=SEED, stratify=test_df["label"]
)

print(f"Amostras de Treino: {len(train_df)}")
print(f"Amostras de Validação: {len(val_df)}")
print(f"Amostras de Teste: {len(test_df)}")

# ----------------------------------------------------------------------
# 3. TOKENIZAÇÃO
# ----------------------------------------------------------------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def preprocess_function(examples):
    return tokenizer(
        examples["resp_text"],
        truncation=True,
        max_length=MAX_LENGTH
    )

# Conversão dos DataFrames Pandas para Hugging Face Dataset
# Garantir que o texto seja sempre string para evitar ArrowTypeError
def preparar_dataset(dataframe):
    dados = dataframe[["resp_text", "label"]].copy()
    dados["resp_text"] = dados["resp_text"].apply(str)
    dados["label"] = dados["label"].astype(int)
    return Dataset.from_pandas(dados, preserve_index=False)

train_dataset = preparar_dataset(train_df)
val_dataset = preparar_dataset(val_df)
test_dataset = preparar_dataset(test_df)
val_dataset = Dataset.from_pandas(val_df[["resp_text", "label"]])
test_dataset = Dataset.from_pandas(test_df[["resp_text", "label"]])

# Aplicar tokenização em lote
train_dataset = train_dataset.map(preprocess_function, batched=True)
val_dataset = val_dataset.map(preprocess_function, batched=True)
test_dataset = test_dataset.map(preprocess_function, batched=True)

# ----------------------------------------------------------------------
# 4. FUNÇÃO DE MÉTRICAS
# ----------------------------------------------------------------------
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    
    acc = accuracy_score(labels, predictions)
    f1_macro = f1_score(labels, predictions, average="macro")
    f1_weighted = f1_score(labels, predictions, average="weighted")
    # QWK penaliza erros entre classes distantes (ex: prever c5 quando é c1)
    qwk = cohen_kappa_score(labels, predictions, weights="quadratic")
    
    return {
        "accuracy": acc,
        "f1_macro": f1_macro,
        "f1_weighted": f1_weighted,
        "qwk": qwk
    }

# ----------------------------------------------------------------------
# 5. INICIALIZAÇÃO DO MODELO
# ----------------------------------------------------------------------
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label=id2label,
    label2id=label_map
)

# ----------------------------------------------------------------------
# 6. CONFIGURAÇÃO E TREINAMENTO (TRAINER)
# ----------------------------------------------------------------------
training_args = TrainingArguments(
    output_dir="./resultados_bertimbau",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    fp16=torch.cuda.is_available(),  # Ativa precisão mista se houver GPU
    logging_steps=100,
    seed=SEED
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
)

# Executar o Treinamento
print("\n--- Iniciando o Treinamento ---")
trainer.train()

# ----------------------------------------------------------------------
# 7. AVALIAÇÃO NO CONJUNTO DE TESTE
# ----------------------------------------------------------------------
print("\n--- Avaliando no Conjunto de Teste ---")
test_results = trainer.evaluate(test_dataset)
print(f"Acurácia no Teste: {test_results['eval_accuracy']:.4f}")
print(f"F1 Macro no Teste: {test_results['eval_f1_macro']:.4f}")
print(f"Quadratic Weighted Kappa (QWK): {test_results['eval_qwk']:.4f}")

# ----------------------------------------------------------------------
# 8. SALVAR O MODELO E TOKENIZER
# ----------------------------------------------------------------------
output_model_dir = "./bertimbau_esic_clarity"
model.save_pretrained(output_model_dir)
tokenizer.save_pretrained(output_model_dir)
print(f"\nModelo salvo com sucesso em: {output_model_dir}")

# ----------------------------------------------------------------------
# 9. FUNÇÃO DE INFERÊNCIA PARA NOVOS TEXTOS
# ----------------------------------------------------------------------
def prever_clareza(texto_resposta):
    model.eval()
    inputs = tokenizer(
        texto_resposta,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LENGTH
    ).to(device)
    
    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
        pred_id = torch.argmax(probs, dim=-1).item()
        
    classe_prevista = id2label[pred_id]
    confianca = probs[0][pred_id].item()
    
    return classe_prevista, confianca

# Exemplo de teste rápido
exemplo_texto = "Prezado cidadão, informamos que os dados solicitados estão disponíveis no portal de transparência através do link http://exemplo.gov.br."
classe, confianca = prever_clareza(exemplo_texto)
print(f"\nExemplo de Predição:")
print(f"Texto: {exemplo_texto[:80]}...")
print(f"Classe Prevista: {classe} (Confiança: {confianca:.2%})")

d:\Users\backup\github\classificador-respostas-esic\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Executando no dispositivo: cpu
Amostras de Treino: 16073
Amostras de Validação: 2009
Amostras de Teste: 2010


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 16478.78it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



--- Iniciando o Treinamento ---


d:\Users\backup\github\classificador-respostas-esic\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss


# Julia-1

In [8]:
from julia import load_model

labels = ["c1", "c234", "c5"]

# Replace these descriptions with the actual meaning of your clarity labels.
label_descriptions = {
    "c1": "A resposta é confusa, pouco clara ou não responde adequadamente à solicitação.",
    "c234": "A resposta tem clareza parcial ou apresenta alguma ambiguidade.",
    "c5": "A resposta é clara, objetiva e responde diretamente à solicitação.",
}

engine = load_model(
    "Julia-1",
    device="cpu",
    strict_encoding=True,
    max_length=8192,
    head_length=512,
    batch_size=16,
)

evaluation_df = df.head(200).copy()

rows = [
    {
        "state": str(text),
        "question": "Qual é o nível de clareza desta resposta?",
        "options": [label_descriptions[label] for label in labels],
    }
    for text in evaluation_df["resp_text"].fillna("")
]

results = engine.predict(rows)

evaluation_df["julia_prediction"] = [
    labels[result["index"]]
    for result in results
]

evaluation_df["julia_confidence"] = [
    max(result["probabilities"])
    for result in results
]

print(classification_report(
    evaluation_df["clarity"],
    evaluation_df["julia_prediction"],
    labels=labels,
    zero_division=0,
))

print(confusion_matrix(
    evaluation_df["clarity"],
    evaluation_df["julia_prediction"],
    labels=labels,
))

evaluation_df[["resp_text", "clarity", "julia_prediction", "julia_confidence"]]

              precision    recall  f1-score   support

          c1       0.29      0.29      0.29        65
        c234       0.42      0.44      0.43        77
          c5       0.18      0.17      0.18        58

    accuracy                           0.32       200
   macro avg       0.30      0.30      0.30       200
weighted avg       0.31      0.32      0.31       200

[[19 22 24]
 [22 34 21]
 [24 24 10]]


,resp_text,clarity,julia_prediction,julia_confidence
0,"Prezado(a) Senhor(a), Esclarecemos que o Se...",c5,c5,0.630030
1,"Prezada cidadã, As informações sobre óbitos ...",c1,c5,0.767889
2,"Prezado Senhor Julio, A Ouvidoria-Geral da P...",c1,c5,0.548829
3,"Prezado(a) Senhor(a), Esclarecemos que o Se...",c234,c5,0.685094
4,"Senhor, O Serviço de Informações ao Cidadão d...",c234,c1,0.769563
...,...,...,...,...
195,"Prezado(a) Senhor(a) Spartaco Angelo, Em ate...",c234,c1,0.654775
196,"Prezado Senhor, Encaminhamos em anexo respost...",c1,c234,0.593555
197,"Prezado cidadão, A Lei nº 12.527/2011, conh...",c1,c1,0.588239
198,"Prezado (a) Senhor (a), Em atenção ao reque...",c234,c5,0.435454
